<a href="https://colab.research.google.com/github/enzozoz/UGS/blob/main/Spark_TPCH_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# !sudo apt update
# !apt-get install openjdk-21-jdk-headless -qq > /dev/null
# Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz
# !wget -q https://www.apache.org/dyn/closer.lua/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz
# !tar xf spark-4.0.1-bin-hadoop3.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-4.0.1-bin-hadoop3"

import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

spark

In [21]:
from pyspark import SparkContext
from pyspark import SparkConf,SparkContext
from pyspark.streaming import StreamingContext
import sys
import requests
from operator import add

from pyspark.sql.types import *
from pyspark.sql import functions as func

from pyspark.sql.functions import lit
from pyspark.sql.functions import udf
from pyspark.sql.functions import *
from pyspark.sql.functions import array

In [22]:

! wget -q https://github.com/kiat/Cloud-Computing/raw/main/Spark-Example-TPCH/tpch_tables_scale_0.1/orders.tbl
! wget -q https://github.com/kiat/Cloud-Computing/raw/main/Spark-Example-TPCH/tpch_tables_scale_0.1/lineitem.tbl
! wget -q https://github.com/kiat/Cloud-Computing/raw/main/Spark-Example-TPCH/tpch_tables_scale_0.1/customer.tbl
! wget -q https://github.com/kiat/Cloud-Computing/raw/main/Spark-Example-TPCH/tpch_tables_scale_0.1/part.tbl
! wget -q https://github.com/kiat/Cloud-Computing/raw/main/Spark-Example-TPCH/tpch_tables_scale_0.1/supplier.tbl
! wget -q https://github.com/kiat/Cloud-Computing/raw/main/Spark-Example-TPCH/tpch_tables_scale_0.1/nation.tbl
! wget -q https://github.com/kiat/Cloud-Computing/raw/main/Spark-Example-TPCH/tpch_tables_scale_0.1/partsupp.tbl
! wget -q https://github.com/kiat/Cloud-Computing/raw/main/Spark-Example-TPCH/tpch_tables_scale_0.1/region.tbl


!ls -la

total 210656
drwxr-xr-x 1 root root     4096 Oct 22 21:10 .
drwxr-xr-x 1 root root     4096 Oct 22 17:51 ..
drwxr-xr-x 4 root root     4096 Oct 17 23:09 .config
-rw-r--r-- 1 root root  2426178 Oct 22 17:55 customer.tbl
-rw-r--r-- 1 root root  2426178 Oct 22 21:10 customer.tbl.1
-rw-r--r-- 1 root root 74247152 Oct 22 17:55 lineitem.tbl
-rw-r--r-- 1 root root 74247152 Oct 22 21:10 lineitem.tbl.1
-rw-r--r-- 1 root root     2257 Oct 22 17:55 nation.tbl
-rw-r--r-- 1 root root     2257 Oct 22 21:10 nation.tbl.1
-rw-r--r-- 1 root root 16893217 Oct 22 17:55 orders.tbl
-rw-r--r-- 1 root root 16893217 Oct 22 21:10 orders.tbl.1
-rw-r--r-- 1 root root 11728237 Oct 22 17:55 partsupp.tbl
-rw-r--r-- 1 root root 11728237 Oct 22 21:10 partsupp.tbl.1
-rw-r--r-- 1 root root  2391154 Oct 22 17:55 part.tbl
-rw-r--r-- 1 root root  2391154 Oct 22 21:10 part.tbl.1
-rw-r--r-- 1 root root      412 Oct 22 17:55 region.tbl
-rw-r--r-- 1 root root      412 Oct 22 21:10 region.tbl.1
drwxr-xr-x 1 root root     4096 O

In [23]:
path = "./"
customer = spark.read.format('csv').options(header='true', inferSchema='true',  sep ="|").load(path+"customer.tbl")
orders = spark.read.format('csv').options(header='true', inferSchema='true',  sep ="|").load(path+"orders.tbl")
lineitems = spark.read.format('csv').options(header='true', inferSchema='true',  sep ="|").load(path+"lineitem.tbl")
part = spark.read.format('csv').options(header='true', inferSchema='true',  sep ="|").load(path+"part.tbl")
supplier = spark.read.format('csv').options(header='true', inferSchema='true',  sep ="|").load(path+"supplier.tbl")
partsupp = spark.read.format('csv').options(header='true', inferSchema='true',  sep ="|").load(path+"partsupp.tbl")
region = spark.read.format('csv').options(header='true', inferSchema='true',  sep ="|").load(path+"region.tbl")
nation = spark.read.format('csv').options(header='true', inferSchema='true',  sep ="|").load(path+"nation.tbl")

In [24]:
print("Customer Table")
customer.show()

print("Orders Table")
orders.show()

print("lineitems Table")
lineitems.show()

print("supplier Table")
supplier.show()

print("partsupp Table")
partsupp.show()

print("region Table")
region.show()

print("nation Table")
nation.show()

Customer Table
+-------+------------------+--------------------+---------+---------------+-------+----------+--------------------+
|CUSTKEY|              NAME|             ADDRESS|NATIONKEY|          PHONE|ACCBATL|MKTSEGMENT|             COMMENT|
+-------+------------------+--------------------+---------+---------------+-------+----------+--------------------+
|      1|Customer#000000001|   IVhzIApeRb ot,c,E|       15|25-989-741-2988| 711.56|  BUILDING|to the even, regu...|
|      2|Customer#000000002|XSTf4,NCwDVaWNe6t...|       13|23-768-687-3665| 121.65|AUTOMOBILE|l accounts. blith...|
|      3|Customer#000000003|        MG9kdTD2WBHm|        1|11-719-748-3364|7498.12|AUTOMOBILE| deposits eat sly...|
|      4|Customer#000000004|         XxVSJsLAGtn|        4|14-128-190-5944|2866.83| MACHINERY| requests. final,...|
|      5|Customer#000000005|KvpyuHCplrB84WgAi...|        3|13-750-942-6364| 794.47| HOUSEHOLD|n accounts will h...|
|      6|Customer#000000006|sKZz0CsnMD7mp4Xd0...|       2

# Develop Spark Code both in RDD and Dataframe.

# Question - 1: Which customers have an account balance greater than 5000, and what are their names, nation keys, and account balances? List the results starting from the highest account balance.



ACCBATL is the account balance in Customers

In [25]:
# RDD solution
customer_rdd = customer.rdd.map(list)

account_balances = customer_rdd.map(lambda x: (x[1],x[3], x[5])).filter(lambda x: float(x[2]) > 5000)

output = account_balances.sortBy(lambda x: float(x[2]), ascending=False).take(20)
for out in output:
  print(out)


# Data Frame Solution

cols = customer.columns

account_balances = (
    customer
    .select(
        customer.NAME,
        customer.NATIONKEY,
        customer.ACCBATL.cast("double").alias("ACCBATL")
    )
    .filter(customer.ACCBATL.cast("double") > 5000)
    .orderBy(customer.ACCBATL.cast("double").desc())
)

account_balances.show(20)


('Customer#000002487', 24, 9999.72)
('Customer#000010494', 24, 9998.87)
('Customer#000013206', 6, 9998.21)
('Customer#000006278', 5, 9996.76)
('Customer#000002760', 16, 9996.34)
('Customer#000007154', 2, 9995.94)
('Customer#000011693', 5, 9995.91)
('Customer#000012437', 1, 9994.84)
('Customer#000008322', 11, 9994.63)
('Customer#000014595', 22, 9994.48)
('Customer#000005126', 19, 9993.81)
('Customer#000013482', 9, 9992.2)
('Customer#000014369', 15, 9991.49)
('Customer#000006087', 17, 9991.42)
('Customer#000009753', 0, 9991.3)
('Customer#000003098', 11, 9989.48)
('Customer#000006432', 22, 9988.42)
('Customer#000000213', 24, 9987.71)
('Customer#000001647', 16, 9987.11)
('Customer#000001701', 3, 9986.13)
+------------------+---------+-------+
|              NAME|NATIONKEY|ACCBATL|
+------------------+---------+-------+
|Customer#000002487|       24|9999.72|
|Customer#000010494|       24|9998.87|
|Customer#000013206|        6|9998.21|
|Customer#000006278|        5|9996.76|
|Customer#0000027

# Task 2: Find out the top-10 Supplier that supplied the most of the products which customers ordered.


Provide your Spark code using Dataframe API.

In [26]:
# supplier_rdd = supplier.rdd.map(list)
# lineitems_rdd = lineitems.rdd.map(list)
# partsupp_rdd = partsupp.rdd.map(list)
# output = lineitems_rdd.map(lambda x: (x[2], x[4])).reduceByKey(lambda x, y: x+y)\
#                                         .join(supplier_rdd.map(lambda x: (x[0], x[1])))\
#                                         .map(lambda x: (x[1][1], x[1][0]))\
#                                         .top(10, lambda x: x[1])


# lineitems.select("")


supplier_totals = (
    lineitems
    .withColumn("QUANTITY", lineitems.QUANTITY.cast("double"))
    .groupBy("SUPPKEY")
    .sum("QUANTITY")
    .withColumnRenamed("sum(QUANTITY)", "TOTAL_QUANTITY")
)

top_suppliers = (
    supplier_totals
    .join(supplier, supplier_totals.SUPPKEY == supplier.SUPPKEY)
    .select(supplier.NAME, supplier_totals.TOTAL_QUANTITY)
    .orderBy(supplier_totals.TOTAL_QUANTITY.desc())
    .limit(10)
)

top_suppliers.show()


+------------------+--------------+
|              NAME|TOTAL_QUANTITY|
+------------------+--------------+
|Supplier#000000074|       18133.0|
|Supplier#000000347|       17775.0|
|Supplier#000000755|       17746.0|
|Supplier#000000736|       17508.0|
|Supplier#000000102|       17499.0|
|Supplier#000000609|       17488.0|
|Supplier#000000474|       17258.0|
|Supplier#000000154|       17214.0|
|Supplier#000000764|       17182.0|
|Supplier#000000944|       17125.0|
+------------------+--------------+



# Task 3: Find out the top-10 customers who have ordered the most distinct number of products.  
Provide your Spark code using Dataframe API.

In [27]:
import pyspark.sql.functions as F


order_part = lineitems.select("ORDERKEY", "PARTKEY")
order_customer = orders.select("ORDERKEY", "CUSTKEY")

customer_part_joined = order_part.join(order_customer, order_part.ORDERKEY == order_customer.ORDERKEY)
customer_part= customer_part_joined.select("CUSTKEY","PARTKEY" )


def my_add(x, y):
    x.add(y)
    return x

# RDD Solution
result = customer_part.rdd.map(list).aggregateByKey(set() , my_add , lambda x, y: x.union( y))
out = result.map(lambda x : (x[0], len(x[1]))).top(10, lambda x: x[1])

for item in out:
  print(item)
# Dataframe Solution
top_customers = (
    customer_part
    .groupBy("CUSTKEY")                                       # group rows by customer to part
    .agg(F.countDistinct("PARTKEY").alias("DISTINCT_PARTS"))  # count unique + name distinct
    .orderBy(F.col("DISTINCT_PARTS").desc())                  # sort by distinct count (highest first)
    .limit(10)                                                # top 10
)

top_customers.show()




(8362, 155)
(346, 153)
(11998, 148)
(9454, 148)
(14707, 148)
(14398, 146)
(10354, 141)
(547, 141)
(85, 141)
(907, 140)
+-------+--------------+
|CUSTKEY|DISTINCT_PARTS|
+-------+--------------+
|   8362|           155|
|    346|           153|
|   9454|           148|
|  11998|           148|
|  14707|           148|
|  14398|           146|
|    547|           141|
|     85|           141|
|  10354|           141|
|    907|           140|
+-------+--------------+





```
# This is formatted as code
```

# Task 4 : Find the nations that have the top-3 highest total sales in this dataset.

In [29]:
# Add your code here !
# lineitems df => [SUPPKEY, PRICE]

nation_sales = (
    lineitems.select("SUPPKEY", col("EXTENDEDPRICE").alias("PRICE"))
    .join(supplier.select("SUPPKEY", "NATIONKEY"), "SUPPKEY")
    .join(nation.select("NATIONKEY", col("NAME").alias("NATION_NAME")), "NATIONKEY")
    .groupBy("NATION_NAME").agg(F.sum("PRICE").alias("TOTAL_SALES"))
    .orderBy(col("TOTAL_SALES").desc())
    .limit(3)
  )

nation_sales.show()


+------------+-------------------+
| NATION_NAME|        TOTAL_SALES|
+------------+-------------------+
|       CHINA|1.134048227300004E9|
|     GERMANY|1.081008329249999E9|
|SAUDI ARABIA|1.031141692089998E9|
+------------+-------------------+



# Task 5: Implement a query that analyzes the LINEITEM table to produce a summary report of sales activity.

The task is to filter the records in the LINEITEM table to include only those items whose shipping date is earlier than or equal to a specified date. After filtering, group the remaining records by the attributes return flag (l_returnflag) and line status (l_linestatus).

For each group, compute the following aggregate values:

*  Total quantity of items ordered
*  Total base price (sum of l_extendedprice)
*  Total discounted price
*  Total charge (including discount and tax)
*  Average quantity
*  Average price
*  Average discount
*  Number of line items in the group

Finally, sort the results by return flag and line status. The output should provide a summarized view of sales performance based on order status and return information.

In [30]:
import pyspark.sql.functions as F

ship_cutoff = "1998-09-02"

li = (
    lineitems.withColumn("QUANTITY", F.col("QUANTITY").cast("double"))
    .withColumn("EXTENDEDPRICE", F.col("EXTENDEDPRICE").cast("double"))
    .withColumn("DSICOUNT", F.col("DISCOUNT").cast("double"))
    .withColumn("TAX", F.col("TAX").cast("double"))
    .withColumn("SHIPDATE", F.to_date(F.col("SHIPDATE")))
    )

pricing = (
    li.filter(F.col("SHIPDATE") <= F.lit(ship_cutoff).cast("date"))
    .groupBy("RETURNFLAG", "LINESTATUS")
    .agg(
        F.sum("QUANTITY").alias("SUM_QTY"),
        F.sum("EXTENDEDPRICE").alias("SUM_BASE_PRICE"),
        F.sum(F.col("EXTENDEDPRICE") * (1 - F.col("DISCOUNT"))).alias("SUM_DISC_PRICE"),
        F.sum(F.col("EXTENDEDPRICE") * (1 - F.col("DISCOUNT")) * (1 + F.col("TAX"))).alias("SUM_CHARGE"),
        F.avg("QUANTITY").alias("AVG_QTY"),
        F.avg("EXTENDEDPRICE").alias("AVG_PRICE"),
        F.avg("DISCOUNT").alias("AVG_DISC"),
        F.count("*").alias("COUNT_ORDER")
    )
    .orderBy("RETURNFLAG", "LINESTATUS")
)

pricing.show()


+----------+----------+---------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|RETURNFLAG|LINESTATUS|  SUM_QTY|      SUM_BASE_PRICE|      SUM_DISC_PRICE|          SUM_CHARGE|           AVG_QTY|         AVG_PRICE|            AVG_DISC|COUNT_ORDER|
+----------+----------+---------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|         A|         F|3774200.0| 5.320753880690026E9| 5.054096266682833E9| 5.256751331449266E9|25.537587116854997| 36002.12382901432| 0.05014459706341364|     147790|
|         N|         F|  95257.0|1.3373779584000002E8|1.2713237265119989E8|1.3228629122944506E8| 25.30066401062417| 35521.32691633466| 0.04939442231075697|       3765|
|         N|         O|7459297.0|1.051227000889980...| 9.986238338384766E9|1.038557837658544E10|25.545537671232875| 36000.92468801302|0.050095958904163025|     

# Task 6: (Optional)

Implenen a query to perform the following steps:

*  Select parts from the PART table that match a given size and whose type ends with a specified string pattern (for example, "BRASS").

*  Consider only suppliers located in a particular region (for example, EUROPE). The relationship between suppliers and regions should be determined through the SUPPLIER, NATION, and REGION tables.

*  Using the PARTSUPP table, determine the minimum supply cost for each qualifying part among suppliers located in that region.

*  Identify the suppliers who offer that minimum supply cost for the part.

*  The query should return information about the suppliers and parts, including supplier name, account balance, nation name, part key, manufacturer, address, phone number, and supplier comments.

Finally, sort the results by:

*  Supplier account balance (descending)

*  Nation name

*  Supplier name

*  Part key

The goal of this task is to identify the most cost-efficient suppliers for specific parts within a region.